Célula única da Silver

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.process_sigma.bronze_orders")
n_bronze = bronze.count()

# 1) Tipar (try_ = se não der, vira nulo em vez de quebrar)
typed = (
    bronze
    .withColumn("order_date", F.expr("try_to_timestamp(order_date, 'MM/dd/yyyy HH:mm')"))
    .withColumn("ship_date", F.expr("try_to_timestamp(ship_date, 'MM/dd/yyyy HH:mm')"))
    .withColumn("days_scheduled", F.expr("try_cast(days_scheduled AS INT)"))
    .withColumn("days_real", F.expr("try_cast(days_real AS INT)"))
    .withColumn("late_delivery_risk", F.expr("try_cast(late_delivery_risk AS INT)"))
)

# 2) Calcular
calc = (
    typed
    .withColumn("delay_days", F.col("days_real") - F.col("days_scheduled"))
    .withColumn("is_late", F.col("delay_days") > 0)
    .withColumn("period", F.date_format("order_date", "yyyy-MM"))
    .withColumn("damage_flag", F.lit(False))
    .withColumn("return_flag", F.lit(False))
)

# 3) Limpar
valid = calc.dropna(subset=["order_id", "order_date", "delay_days"])
n_valid = valid.count()
silver = valid.dropDuplicates(["order_id"])
n_silver = silver.count()

print(f"Bronze: {n_bronze} | após nulos: {n_valid} | após duplicados: {n_silver}")

silver.write.mode("overwrite").saveAsTable("workspace.process_sigma.silver_orders")

Bronze: 500 | após nulos: 500 | após duplicados: 500


Conferir

In [0]:
s = spark.table("workspace.process_sigma.silver_orders")
s.printSchema()
s.select("order_id", "order_date", "days_scheduled", "days_real",
         "delay_days", "is_late", "period").show(5, truncate=False)
print(s.count(), s.filter("is_late").count())

root
 |-- order_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- ship_date: timestamp (nullable = true)
 |-- days_scheduled: integer (nullable = true)
 |-- days_real: integer (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_delivery_risk: integer (nullable = true)
 |-- order_region: string (nullable = true)
 |-- order_country: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- source: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)
 |-- period: string (nullable = true)
 |-- damage_flag: boolean (nullable = true)
 |-- return_flag: boolean (nullable = true)

+--------+-------------------+--------------+---------+----------+-------+-------+
|order_id|order_date         |days_scheduled|days_real|delay_days|is_late|period |
+--------+-------------------+--------------+---------+----------+-------+-------+
|1